In [13]:
import sqlite3
import numpy as np
import pandas as pd
import xarray as xr
import os
import sys

sys.path.append(os.path.abspath('../../'))
from src.hydro_utils import calculate_grid_cell_areas

In [14]:
mask_path = "/Users/ljob/Desktop/cnbs-predictor/data/input/SFS_GL_mask.nc"
database_path = "/Users/ljob/Desktop/Data/SFS_reforecast/sfs_reforecast_data.db"

variable_name = "tmp2m"#"avg_slhtfsfc" #"tmp2m" "pratesfc"
long_name = "air_temperature" #"latent_heat_flux" #"precipitation" "air_temperature"

months = ["04"]

start_date = "1991-04-01"
end_date = "1992-04-01"

In [15]:
mask_ds = xr.open_dataset(mask_path)

mask_variables = [
    var
    for var in mask_ds.data_vars
    if var.endswith("_lake") or var.endswith("_land")
]

print("Processing masks:")
for mask_var in mask_variables:
    print(f"  {mask_var}")

mask_lat = mask_ds["lat"].values
mask_lon = mask_ds["lon"].values

Processing masks:
  superior_lake
  superior_land
  michigan-huron_lake
  michigan-huron_land
  erie_lake
  erie_land
  ontario_lake
  ontario_land


In [16]:
# Calculate grid-cell area for area-weighted lake/land averages.
area = calculate_grid_cell_areas(mask_lon, mask_lat)

In [17]:
print()
print("Opening database...")

conn = sqlite3.connect(database_path)

cursor = conn.cursor()

# Create the table if it doesn't exist
cursor.execute(
    """
    CREATE TABLE IF NOT EXISTS reforecast_data (
        init_time TEXT,
        member INTEGER,
        lead INTEGER,
        valid_time TEXT,
        lake TEXT,
        surface_type TEXT,
        component TEXT,
        value REAL
    )
    """
)

# Create Index
cursor.execute(
    """
    CREATE INDEX IF NOT EXISTS idx_reforecast_lookup
    ON reforecast_data (
        init_time,
        member,
        lead,
        lake,
        surface_type,
        component
    )
    """
)

conn.commit()


Opening database...


In [18]:
for month in months:

    zarr_path = f"s3://noaa-oar-sfsdev-pds/experiments/beta1/reforecast/{month}/atm_monthly.zarr"

    try:
        ds = xr.open_zarr(
        zarr_path,
        storage_options={"anon": True}
        )

        variable = ds[variable_name]

        for init_time in variable.init.sel(init=slice(start_date, end_date)).values:

            # -----------------------------------------------------------
            # Convert initialization time to pandas Timestamp
            # -----------------------------------------------------------

            init_timestamp = pd.Timestamp(init_time)
            seconds_in_month = init_timestamp.days_in_month * 86400

            print()
            print("=" * 70)
            print(
                f"Processing initialization: "
                f"{init_timestamp:%Y-%m-%d}"
            )
            print("=" * 70)

            for member in variable.member.values:

                member = int(member)

                print()
                print(f"  Member: {member}")
                for lead in variable.lead.values:

                    lead = int(lead)
                    valid_time = (
                        init_timestamp
                        + pd.DateOffset(months=lead)
                    )


                    print(
                        f"    Lead {lead:02d} "
                        f"-> {valid_time:%Y-%m}"
                    )

                    # ===================================================
                    # Select temperature field
                    # ===================================================

                    tmp = variable.sel(
                        init=init_time,
                        member=member,
                        lead=lead,
                    )

                    # ===================================================
                    # Cut temperature field to mask extent
                    # ===================================================

                    tmp_cut = tmp.sel(
                        lat=slice(
                            mask_lat.min(),
                            mask_lat.max(),
                        ),
                        lon=slice(
                            mask_lon.min(),
                            mask_lon.max(),
                        ),
                    )

                    tmp_values = tmp_cut.values

                    # ===================================================
                    # Loop through lake/land masks
                    # ===================================================

                    for mask_var in mask_variables:

                        lake, surface_type = mask_var.rsplit(
                            "_",
                            1,
                        )

                        # ------------------------------------------------
                        # Get fractional mask
                        # ------------------------------------------------

                        mask = mask_ds[mask_var].values

                        # ------------------------------------------------
                        # Make sure mask and temperature have compatible
                        # dimensions.
                        # ------------------------------------------------

                        if mask.shape != tmp_values.shape:

                            raise ValueError(
                                f"Mask shape does not match temperature "
                                f"shape for {mask_var}: "
                                f"mask={mask.shape}, "
                                f"temperature={tmp_values.shape}"
                            )
                        # LATENT HEAT FLUX
                        #tmp_avg = (
                        #    np.nansum(tmp_values * mask * area)
                        #    / np.nansum(mask * area)
                        #)
                        
                        # NEEDED FOR PRECIPITATION RATE CONVERSION FROM RATE TO TOTAL AMOUNT OVER THE MONTH
                        #weighted_tmp_rate = (
                        #    np.nansum(tmp_values * mask * area)
                        #    / np.nansum(mask * area)   
                        #)
                        
                        #tmp_avg = weighted_tmp_rate * seconds_in_month
                        # ------------------------------------------------

                        # START OF TEMPERATURE AVERAGING

                        # ------------------------------------------------
                        # Identify valid cells.
                        #
                        # Exclude:
                        #
                        #   - NaN mask cells
                        #   - NaN temperature cells
                        # ------------------------------------------------

                        valid = (
                            np.isfinite(mask)
                            & np.isfinite(tmp_values)
                        )

                        # ------------------------------------------------
                        # Extract valid values and weights
                        # ------------------------------------------------

                        values = tmp_values[valid]
                        weights = mask[valid]
                        
                        # ------------------------------------------------
                        # Calculate weighted mean
                        #
                        # Fractional masks are handled here:
                        #
                        # 1.00 = full cell
                        # 0.75 = 75% of cell
                        # 0.50 = 50% of cell
                        # 0.25 = 25% of cell
                        #
                        # NaN cells are excluded.
                        # ------------------------------------------------

                        if (
                            len(values) == 0
                            or np.sum(weights) == 0
                        ):

                            tmp_avg = np.nan

                        else:

                            tmp_avg = np.average(
                                values,
                                weights=weights,
                            )

                        # END OF TEMPERATURE AVERAGING


                        # ------------------------------------------------
                        # Insert result into database
                        # ------------------------------------------------

                        cursor.execute(
                            """
                            INSERT INTO reforecast_data (
                                init_time,
                                member,
                                lead,
                                valid_time,
                                lake,
                                surface_type,
                                component,
                                value
                            )
                            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
                            """,
                            (
                                init_timestamp.strftime(
                                    "%Y-%m-%d"
                                ),
                                member,
                                lead,
                                valid_time.strftime(
                                    "%Y-%m-%d"
                                ),
                                lake,
                                surface_type,
                                long_name,
                                (
                                    float(tmp_avg)
                                    if np.isfinite(tmp_avg)
                                    else None
                                ),
                            ),
                        )


                # -------------------------------------------------------
                # Commit after each ensemble member
                #
                # This prevents a large number of uncommitted database
                # transactions from accumulating.
                # -------------------------------------------------------

                conn.commit()

    except Exception as e:

        print()
        print("=" * 70)
        print(f"Error processing month {month}: {e}")
        print("=" * 70)
        print()

        continue

# ===============================================================
# Close database
# ===============================================================

conn.close()

print()
print("=" * 70)
print("Processing complete.")
print("=" * 70)
print()
print(f"Database: {database_path}")            



Processing initialization: 1991-04-01

  Member: 0
    Lead 00 -> 1991-04
    Lead 01 -> 1991-05
    Lead 02 -> 1991-06
    Lead 03 -> 1991-07
    Lead 04 -> 1991-08
    Lead 05 -> 1991-09
    Lead 06 -> 1991-10
    Lead 07 -> 1991-11
    Lead 08 -> 1991-12
    Lead 09 -> 1992-01
    Lead 10 -> 1992-02
    Lead 11 -> 1992-03

  Member: 1
    Lead 00 -> 1991-04
    Lead 01 -> 1991-05
    Lead 02 -> 1991-06
    Lead 03 -> 1991-07
    Lead 04 -> 1991-08
    Lead 05 -> 1991-09
    Lead 06 -> 1991-10
    Lead 07 -> 1991-11
    Lead 08 -> 1991-12
    Lead 09 -> 1992-01
    Lead 10 -> 1992-02
    Lead 11 -> 1992-03

  Member: 2
    Lead 00 -> 1991-04
    Lead 01 -> 1991-05
    Lead 02 -> 1991-06
    Lead 03 -> 1991-07
    Lead 04 -> 1991-08
    Lead 05 -> 1991-09
    Lead 06 -> 1991-10
    Lead 07 -> 1991-11
    Lead 08 -> 1991-12
    Lead 09 -> 1992-01
    Lead 10 -> 1992-02
    Lead 11 -> 1992-03

  Member: 3
    Lead 00 -> 1991-04
    Lead 01 -> 1991-05
    Lead 02 -> 1991-06
    Lead 03 